In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS cicd_dev.silver.silver_audit (
    table_name STRING,
    load_time TIMESTAMP,
    record_count BIGINT,
    status STRING
)
""")

In [0]:
from pyspark.sql.functions import *


def write_silver_table(
    source_table,
    target_table,
    transformation_logic
):

    try:

        df = spark.table(source_table)

        transformed_df = transformation_logic(df)

        transformed_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

        record_count = transformed_df.count()

        audit_df = spark.createDataFrame(
            [(target_table, record_count, "SUCCESS")],
            ["table_name","record_count","status"]
        ).select(
            "*",
            current_timestamp().alias("load_time")
        )

        audit_df.write.mode("append").saveAsTable(
            "cicd_dev.silver.silver_audit"
        )

    except Exception as e:

        audit_df = spark.createDataFrame(
            [(target_table, 0, f"FAILED : {str(e)}")],
            ["table_name","record_count","status"]
        ).select(
            "*",
            current_timestamp().alias("load_time")
        )

        audit_df.write.mode("append").saveAsTable(
            "cicd_dev.silver.silver_audit"
        )

In [0]:
def customer_transform(df):

    return (
        df
        .dropDuplicates(["customer_id"])
        .filter(col("customer_id").isNotNull())
        .withColumn(
            "city",
            upper(col("city"))
        )
    )

In [0]:
write_silver_table(
    "cicd_dev.bronze.customers_raw",
    "cicd_dev.silver.customers",
    customer_transform
)

In [0]:
%sql
SELECT COUNT(*)
FROM cicd_dev.silver.customers;

In [0]:
%sql
SELECT *
FROM cicd_dev.silver.silver_audit
ORDER BY load_time DESC;